In [115]:
import pandas as pd
import pickle
import json

df = pd.read_csv('anilist_anime_data_complete.csv')

In [112]:
df = df[df['popularity'] > 30000]
df = df[df['isAdult'] == False]
df = df.reset_index()

In [113]:
df = df[['title_english', 'episodes', 'duration', 'genres', 'tags']]
b = df.copy()

In [114]:
df.iloc[645]

title_english                                                  NaN
episodes                                                      38.0
duration                                                      24.0
genres                               ["Comedy", "Drama", "Sports"]
tags             [{"id": 151, "name": "Cycling", "description":...
Name: 645, dtype: object

In [110]:
import requests
import json
import pandas as pd

query = '''
query ($type: MediaType!, $userName: String!) {
  MediaListCollection(type: $type, userName: $userName) {
    lists {
      name
      entries {
        id
        media {
          id
          title {
            english
          }
          episodes
          duration
          genres
          tags{
              name
              rank
          }
        }
      }
    }
  }
}
'''

variables = {
    'type': 'ANIME',
    'userName':'aryantestlist'
}

url = 'https://graphql.anilist.co'

response = requests.post(url, json={'query': query, 'variables': variables})

json_data = json.loads(response.content)
cleaned = []
for j in json_data['data']['MediaListCollection']['lists']:
    print(j['name'])
    if j['name'] != "Dropped" and j['name'] != "Planning":
        for show in j['entries']:
            show_data = []
            show_data.append(str(show['media']['title']['english']))
            show_data.append(float(show['media']['episodes']))
            show_data.append(float(show['media']['duration']))
            show_data.append(json.dumps(show['media']['genres']))
            tags = []
            for tag in show['media']['tags']:
                if tag['rank'] > 65:
                    tags.append({"name": tag['name'], "rank": tag['rank'], "isAdult":False})
                    #tags.append([tag['name'], tag['rank']])
            show_data.append(json.dumps(tags))
            #show_data.append(float(show['media']['averageScore']))
            #show_data.append(int(show['media']['popularity']))
            cleaned.append(show_data)
data = pd.DataFrame(data=cleaned, columns=['title_english', 'episodes', 'duration', 'genres', 'tags'])
df = df.append(data)
df.drop_duplicates(subset=['title_english'], inplace=True, keep='last')
df.reset_index(inplace=True)

Completed


KeyError: 'english'

In [87]:
df = df.drop(columns=['index'])


In [88]:
df['genres']

0          ["Action", "Drama", "Slice of Life", "Sports"]
1                                 ["Adventure", "Comedy"]
2                            ["Drama", "Mecha", "Sci-Fi"]
3                       ["Adventure", "Drama", "Romance"]
4            ["Action", "Adventure", "Comedy", "Mystery"]
                              ...                        
2140                                 ["Action", "Comedy"]
2141                 ["Drama", "Romance", "Supernatural"]
2142    ["Action", "Adventure", "Drama", "Fantasy", "P...
2143    ["Action", "Adventure", "Drama", "Fantasy", "P...
2144                        ["Action", "Drama", "Sports"]
Name: genres, Length: 2145, dtype: object

In [89]:
data['genres']

0                         ["Comedy", "Drama", "Sports"]
1                         ["Comedy", "Drama", "Sports"]
2                         ["Comedy", "Drama", "Sports"]
3           ["Comedy", "Slice of Life", "Supernatural"]
4           ["Comedy", "Slice of Life", "Supernatural"]
5                                  ["Comedy", "Sports"]
6                   ["Action", "Drama", "Supernatural"]
7                   ["Action", "Drama", "Supernatural"]
8                   ["Action", "Drama", "Supernatural"]
9     ["Mystery", "Psychological", "Supernatural", "...
10      ["Drama", "Romance", "Sci-Fi", "Slice of Life"]
11            ["Action", "Drama", "Fantasy", "Mystery"]
12            ["Action", "Drama", "Fantasy", "Mystery"]
13            ["Action", "Drama", "Fantasy", "Mystery"]
14            ["Action", "Drama", "Fantasy", "Mystery"]
15            ["Action", "Drama", "Fantasy", "Mystery"]
16    ["Action", "Drama", "Fantasy", "Mystery", "Psy...
17    ["Action", "Drama", "Fantasy", "Mystery", 

In [90]:
for tags in range(len(df['tags'].apply(json.loads))):
    for tag in df['tags'].apply(json.loads)[tags]:
        if tag['rank'] > 75 and tag['isAdult'] == False:
            if tag['name'] not in df.columns:
                df[tag['name']] = 0
            df[tag['name']][tags] = tag['rank']

C:\Users\Aryan\AppData\Local\Temp\ipykernel_3628\1220729427.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[tag['name']][tags] = tag['rank']
C:\Users\Aryan\AppData\Local\Temp\ipykernel_3628\1220729427.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[tag['name']][tags] = tag['rank']
C:\Users\Aryan\AppData\Local\Temp\ipykernel_3628\1220729427.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[tag['name']][tags] = tag['ra

In [91]:
for genres in range(len(df['genres'].apply(json.loads))):
    for genre in df['genres'].apply(json.loads)[genres]:
            if genre not in df.columns:
                df[genre] = 0
            df[genre][genres] = 1

C:\Users\Aryan\AppData\Local\Temp\ipykernel_3628\906486332.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[genre] = 0
C:\Users\Aryan\AppData\Local\Temp\ipykernel_3628\906486332.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[genre][genres] = 1
C:\Users\Aryan\AppData\Local\Temp\ipykernel_3628\906486332.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newfr

In [92]:
df.fillna(0, inplace=True)

In [93]:
df.head()

,title_romaji,episodes,duration,genres,tags,Boxing,Coming of Age,Delinquents,Tragedy,Male Protagonist,...,Romance,Mystery,Fantasy,Horror,Supernatural,Psychological,Music,Ecchi,Mahou Shoujo,Thriller
0,Ashita no Joe,79.0,25.0,"[""Action"", ""Drama"", ""Slice of Life"", ""Sports""]","[{""id"": 169, ""name"": ""Boxing"", ""description"": ...",90,90,87,83,80,...,0,0,0,0,0,0,0,0,0,0
1,Lupin III,23.0,25.0,"[""Adventure"", ""Comedy""]","[{""id"": 50, ""name"": ""Seinen"", ""description"": ""...",0,0,0,0,80,...,0,0,0,0,0,0,0,0,0,0
2,Kidou Senshi Gundam,43.0,25.0,"[""Drama"", ""Mecha"", ""Sci-Fi""]","[{""id"": 111, ""name"": ""War"", ""description"": ""Pa...",0,80,0,80,0,...,0,0,0,0,0,0,0,0,0,0
3,Versailles no Bara,40.0,23.0,"[""Adventure"", ""Drama"", ""Romance""]","[{""id"": 25, ""name"": ""Historical"", ""description...",0,0,0,97,0,...,1,0,0,0,0,0,0,0,0,0
4,Lupin III: Cagliostro no Shiro,1.0,100.0,"[""Action"", ""Adventure"", ""Comedy"", ""Mystery""]","[{""id"": 648, ""name"": ""Crime"", ""description"": ""...",0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [94]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [95]:
cosine_sim = linear_kernel(df.iloc[:, 7:], df.iloc[:, 7:])

In [96]:
indices = pd.Series(df.index, index=df['title_english']).drop_duplicates()

In [97]:
def get_recommendations(title, a_list, cosine_sim=cosine_sim, num_recommend = 10):
    if title in df['title_english'].values:
        num_recd = 0
        raw_recd = 1
        top_similar = []
        idx = indices[title]
        sim_scores = list(enumerate(cosine_sim[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        while num_recd <= num_recommend:
            top_rec = sim_scores[raw_recd]
            if df['title_english'].iloc[top_rec[0]] not in a_list and "Season" not in df['title_english'].iloc[top_rec[0]]:
            #print(df['title_romaji'].iloc[top_rec[0]])
                top_similar.append(sim_scores[raw_recd])
                num_recd += 1
            raw_recd += 1
        movie_indices = [i[0] for i in top_similar]
        return df['title_english'].iloc[movie_indices]

In [98]:
get_recommendations('Haikyuu!!', cleaned, num_recommend = 20)

2106                                            Haikyuu!!
1391                                 Haikyuu!! TO THE TOP
2108    Haikyuu!!: Karasuno Koukou VS Shiratorizawa Ga...
1897                     Haikyuu!!: Gomi Suteba no Kessen
1864                         Tokyo Revengers: Tenjiku-hen
2144                                            Blue Lock
1534                                   Shakunetsu Kabaddi
1933                             Blue Lock VS. U-20 JAPAN
650                                        Diamond no Ace
1421                                Haikyuu!! Riku VS Kuu
622                           Kuroko no Basket 2nd SEASON
1556                                            Bakuten!!
647                                                 Free!
91                                           Eyeshield 21
661                       HUNTER×HUNTER: THE LAST MISSION
739                                 Free!: Eternal Summer
2111                                     Kuroko no Basket
202           

In [106]:
import requests
import json
query = '''
query ($type: MediaType!, $userName: String!) {
  MediaListCollection(type: $type, userName: $userName) {
    lists {
      name
      entries {
        id
        media {
          title {
            title_english
          }
        }
      }
    }
  }
}
'''

variables = {
    'type': 'ANIME',
    'userName':'aryantestlist'
}

url = 'https://graphql.anilist.co'

response = requests.post(url, json={'query': query, 'variables': variables})

In [107]:
json_data = json.loads(response.content)
anime_list = []
for j in json_data['data']['MediaListCollection']['lists']:
    if j['name'] != "Dropped" and j['name'] != "Planning":
        for show in j['entries']:
            anime_list.append(str(show['media']['title']['english']))

In [108]:
total_recs = {}
for title in anime_list:
    #if title in df['title_romaji'].values:
        print(title)
        recs = list(get_recommendations(title, anime_list, num_recommend = 20))
        for rec in recs:
            if rec not in anime_list:
                if rec in total_recs:
                    total_recs[rec] += recs.index(rec)
                else:
                    total_recs[rec] = recs.index(rec)

Haikyuu!!
Haikyuu!! 2nd Season
Haikyuu!!: Karasuno Koukou VS Shiratorizawa Gakuen Koukou
Saiki Kusuo no Ψ-nan
Saiki Kusuo no Ψ-nan 2
Kuroko no Basket
Jujutsu Kaisen
Jujutsu Kaisen 2nd Season
Jujutsu Kaisen: Shimetsu Kaiyuu - Zenpen
DEATH NOTE
Plastic Memories
Shingeki no Kyojin
Shingeki no Kyojin Season 2
Shingeki no Kyojin Season 3
Shingeki no Kyojin Season 3 Part 2
Shingeki no Kyojin: The Final Season
Shingeki no Kyojin: The Final Season Part 2
Shingeki no Kyojin: The Final Season - Kanketsu-hen Zenpen
Shingeki no Kyojin: The Final Season - Kanketsu-hen Kouhen
Ansatsu Kyoushitsu
Ansatsu Kyoushitsu 2nd Season
Cyberpunk: Edgerunners
Hyouka
Steins;Gate
Shoushimin Series
Shoushimin Series 2nd Season
86: Eighty Six
86: Eighty Six Part 2
Shigatsu wa Kimi no Uso
Ao no Hako
Angel Beats!
Charlotte
Chainsaw Man
Chainsaw Man: Reze-hen
Kill Ao
Kimi no Na wa.
Re:Zero kara Hajimeru Isekai Seikatsu
Re:Zero kara Hajimeru Isekai Seikatsu 2nd Season
Blue Lock


In [109]:
dict(sorted(total_recs.items(), key=lambda item: item[1], reverse=True))

{'Summer Time Render': 152,
 'Kenpuu Denki Berserk': 107,
 'Kimetsu no Yaiba: Mugenjou-hen Movie 1 - Akaza Sairai': 94,
 'Tokyo Revengers: Tenjiku-hen': 88,
 'Kaguya-sama wa Kokurasetai: Ultra Romantic': 88,
 'PLUTO': 87,
 'Kaguya-sama wa Kokurasetai?: Tensaitachi no Renai Zunousen': 86,
 'Another': 85,
 'Tokyo Revengers': 84,
 'Boku no Hero Academia 6': 83,
 'CLANNAD': 79,
 'Ima, Soko ni Iru Boku': 79,
 'Code Geass: Hangyaku no Lelouch R2': 78,
 'Enen no Shouboutai: San no Shou': 73,
 'MASHLE': 70,
 'VINLAND SAGA': 70,
 'Shinsekai yori': 68,
 'SANDA': 67,
 'Undead Unluck': 65,
 'Katekyou Hitman REBORN!': 64,
 'JoJo no Kimyou na Bouken: Stardust Crusaders - Egypt-hen': 64,
 'JIN-ROH': 64,
 'MONSTER': 63,
 'Eyeshield 21': 62,
 'Kimi no Koto ga Dai Dai Dai Dai Daisuki na 100-nin no Kanojo': 62,
 'Gachiakuta': 62,
 'Terra Formars': 57,
 'HUNTER×HUNTER (2011)': 55,
 'Shingeki no Kyojin Gaiden: Kuinaki Sentaku': 54,
 'Kidou Senshi Gundam: Tekketsu no Orphans': 54,
 'Tokyo Revengers: Seiya K